In [1]:
import pandas as pd
import tqdm, os, glob
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.85,1.87,1.87,2.04,ð,ɛ,ð,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,2.04,2.07,ɛ,n,ð_ɛ,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,2.09,2.14,ð,ʌ,ð,the,1,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,2.18,2.24,n,ɔ,n,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,2.24,2.39,ɔ,ɹ,n_ɔ,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
26851,52.92,52.97,52.97,53.01,æ,ŋ,l_æ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26852,52.97,53.01,53.01,53.04,ŋ,g,l_æ_ŋ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26853,53.01,53.04,53.04,53.10,g,w,l_æ_ŋ_g,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26854,53.04,53.10,53.10,53.28,w,ɨ,l_æ_ŋ_g_w,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [ ]:
tokens = {
    "start": [],
    "end": [],
    "syllable": [],
    "art_f": [],
    "ac_f": [],
    "subject": [],
}

metadata = set(
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable1_start"], df["syllable1_end"], df["syllable1"], df["art_f"], df["ac_f"], df["subject"])] + 
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable2_start"], df["syllable2_end"], df["syllable2"], df["art_f"], df["ac_f"], df["subject"])]
)

metadata = sorted(list(metadata), key=lambda x: (x[3], x[4], x[5]))

for start, end, target, art_f, ac_f, subject in metadata:
    tokens["start"].append(start)
    tokens["end"].append(end)
    tokens["syllable"].append(target)
    tokens["art_f"].append(art_f)
    tokens["ac_f"].append(ac_f)
    tokens["subject"].append(subject)

In [5]:
token_df = pd.DataFrame(tokens)

In [6]:
for subject in token_df["subject"].unique():
    os.makedirs(f"aemri75anno_scalers", exist_ok=True)
    if os.path.exists(f"aemri75anno_scalers/{subject}.pkl"):
        continue

    subject_token_df = token_df[token_df["subject"]==subject]

    data = {}

    for start, end, art_f in tqdm.tqdm(zip(subject_token_df["start"], subject_token_df["end"], subject_token_df["art_f"]), total=len(subject_token_df)):
        features = get_aemri75anno_articulatory_features(art_f, start, end)
        if features is None:
            continue
        features = features.reshape(features.shape[0], -1)
        for idx in range(features.shape[1]):
            if idx not in data:
                data[idx] = []
            data[idx].append(features[:, idx])

    scalers = {}
    for idx in tqdm.tqdm(data, total=len(data)):
        data[idx] = np.concatenate(data[idx]).reshape(-1, 1)
        scaler = StandardScaler()
        scaler.fit(data[idx])
        scalers[idx] = scaler

    with open(f"aemri75anno_scalers/{subject}.pkl", "wb") as f:
        pickle.dump(scalers, f)    

100%|██████████| 348/348 [00:00<00:00, 617.82it/s]


In [7]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"aemri75anno_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder

In [9]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=1024,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aemri75anno_articulatory_features(art_f, start, end, scalers=scalers[subject])

    if articulatory_features is None:
        continue

    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 174, 2)

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_x = art_model(art_batch)
    art_loss, art_static, art_delta = combined_loss(
        art_out, art_batch
    )

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_x)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aemri75anno/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75anno/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aemri75anno/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75anno/art_model_training_log.csv", index=False)

300: ART 0.3629:   3%|▎         | 1204/37849 [02:45<1:23:43,  7.29it/s]

[ART] latent var mean=1.0379e+03, min=1.4311e-01
[ART] input std=7.2114e-01, recon std=2.2517e-01


RuntimeError: Parent directory ckpts/aeaemri75anno does not exist.

In [ ]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

999: AC 122.3411:   7%|▋         | 3980/56882 [00:25<05:00, 175.76it/s]

[AC ] latent var mean=2.8484e+02, min=2.2508e-04
[AC ] input std=1.0931e+01, recon std=5.1379e+00


1999: AC 191.7450:  14%|█▍        | 7978/56882 [00:47<04:37, 176.35it/s]

[AC ] latent var mean=8.0272e+02, min=6.3266e-04
[AC ] input std=1.4378e+01, recon std=6.7728e+00


2999: AC 101.8416:  21%|██        | 11977/56882 [01:10<04:11, 178.76it/s]

...

12999: AC 118.8315:  91%|█████████▏| 51993/56882 [04:50<00:27, 180.78it/s]

[AC ] latent var mean=3.2173e+05, min=3.4532e-01
[AC ] input std=1.2230e+01, recon std=6.3713e+00


13999: AC 99.5192:  98%|█████████▊| 55976/56882 [05:12<00:04, 215.38it/s] 

[AC ] latent var mean=1.3194e+06, min=7.2117e-01
[AC ] input std=1.3624e+01, recon std=6.8483e+00


14220: AC 103.5079: 100%|██████████| 56882/56882 [05:17<00:00, 179.04it/s]
